# 导入 (import)

In [120]:
import pandas as pd

'''
处理 jsonl 为 pandas dataframe
Process jsonl to pandas dataframe
'''
words_df = pd.read_json("./crawler_output.jl", orient="records", lines=True)
words_df.head()

,word,part_of_speech,phonetic_symbol,definitions
0,RN,noun,"[/ˌɑːrˈen/, /ˌɑːrˈen/]","[{'cefr': '[ C ] US', 'definition': {'en': 'a ..."
1,R.N.,noun [ C ],"[/ˌɑːrˈen/, /ˌɑːrˈen/]","[{'cefr': '', 'definition': {'en': 'abbreviati..."
2,riyal,noun [ C ],"[/riˈɑːl/, /riːˈjɑːl/]","[{'cefr': '', 'definition': {'en': 'the standa..."
3,rivulet,noun [ C ],"[/ˈrɪv.jə.lət/, /ˈrɪv.jə.lət/]","[{'cefr': '', 'definition': {'en': 'a very sma..."
4,riviera,noun [ C ],"[/rɪv.iˈeə.rə/, /rɪv.iˈer.ə/]","[{'cefr': '', 'definition': {'en': 'an area of..."


# 赋予类型

In [121]:
words_df = words_df.astype({"word": str, "part_of_speech": str, "phonetic_symbol": object, "definitions": object})
words_df.head()

,word,part_of_speech,phonetic_symbol,definitions
0,RN,noun,"[/ˌɑːrˈen/, /ˌɑːrˈen/]","[{'cefr': '[ C ] US', 'definition': {'en': 'a ..."
1,R.N.,noun [ C ],"[/ˌɑːrˈen/, /ˌɑːrˈen/]","[{'cefr': '', 'definition': {'en': 'abbreviati..."
2,riyal,noun [ C ],"[/riˈɑːl/, /riːˈjɑːl/]","[{'cefr': '', 'definition': {'en': 'the standa..."
3,rivulet,noun [ C ],"[/ˈrɪv.jə.lət/, /ˈrɪv.jə.lət/]","[{'cefr': '', 'definition': {'en': 'a very sma..."
4,riviera,noun [ C ],"[/rɪv.iˈeə.rə/, /rɪv.iˈer.ə/]","[{'cefr': '', 'definition': {'en': 'an area of..."


# 展开释义

In [122]:
assert words_df["definitions"].isnull().sum() == 0, "数据处理错误，存在缺失的 definitions！Data processing error, missing definitions!"

words_df = words_df.explode("definitions")
definitions = pd.json_normalize(words_df["definitions"])
definitions.rename(columns={
    "definition.en": "definition_en",
    "definition.zh": "definition_zh"
}, inplace=True)

words_df = pd.concat([words_df.drop(columns=["definitions"]), definitions], axis=1)
words_df = words_df.astype({"definition_en": str, "definition_zh": str})

assert len(definitions) == len(words_df), "数据处理错误，行数不匹配！Data processing error, row count mismatch!"
assert words_df["definition_en"].isnull().sum() == 0, "数据处理错误，存在缺失的 definition_en！Data processing error, missing definition_en!"
assert words_df["definition_zh"].isnull().sum() == 0, "数据处理错误，存在缺失的 definition_zh！Data processing error, missing definition_zh!"
words_df.head()

,word,part_of_speech,phonetic_symbol,cefr,examples,definition_en,definition_zh
0,RN,noun,"[/ˌɑːrˈen/, /ˌɑːrˈen/]",[ C ] US,[{'en': 'An RN makes about $30 an hour in this...,a \n \n registered nurse ; als...,注册护士（可用于姓名后）
0,RN,noun,"[/ˌɑːrˈen/, /ˌɑːrˈen/]",UK,"[{'en': 'Captain H. Doughty, RN', 'zh': '英国皇家海...",abbreviation for Royal Navy: used especially a...,英国皇家海军（尤用于海军军官姓名后，Royal Navy的缩写）
1,R.N.,noun [ C ],"[/ˌɑːrˈen/, /ˌɑːrˈen/]",,[],abbreviation for\n \n registere...,注册护士（registered nurse的缩写）
2,riyal,noun [ C ],"[/riˈɑːl/, /riːˈjɑːl/]",,[],the standard unit of money used in Saudi Arabi...,里亚尔（沙特阿拉伯和卡塔尔的货币单位）
3,rivulet,noun [ C ],"[/ˈrɪv.jə.lət/, /ˈrɪv.jə.lət/]",,[{'en': 'Rivulets of sweat/rain/blood ran down...,a very small stream or flow of liquid,小溪;小河;细流


# 展开例子

In [123]:
words_df = words_df.explode("examples")

# 处理 examples 列，空的例子用 {} 填充，方便继续 json normalize 展开
words_df["examples"] = words_df["examples"].apply(lambda x: x if isinstance(x, dict) else {})
examples = pd.json_normalize(words_df["examples"])
examples.rename(columns={"en": "example_en", "zh": "example_zh"}, inplace=True)

words_df = pd.concat([words_df.drop(columns=["examples"]), examples], axis=1)
words_df = words_df.astype({"example_en": str, "example_zh": str})
words_df.fillna({"example_en": ""}, inplace=True)
words_df.fillna({"example_zh": ""}, inplace=True)

assert len(examples) == len(words_df), "数据处理错误，行数不匹配！Data processing error, row count mismatch!"
assert words_df["example_en"].isnull().sum() == 0, "数据处理错误，存在缺失的 example_en！Data processing error, missing example_en!"
assert words_df["example_zh"].isnull().sum() == 0, "数据处理错误，存在缺失的 example_zh！Data processing error, missing example_zh!"
words_df.head()

,word,part_of_speech,phonetic_symbol,cefr,definition_en,definition_zh,example_en,example_zh
0,RN,noun,"[/ˌɑːrˈen/, /ˌɑːrˈen/]",[ C ] US,a \n \n registered nurse ; als...,注册护士（可用于姓名后）,An RN makes about $30 an hour in this city.,在这座城市里一名注册护士的时薪约为30美元。
0,RN,noun,"[/ˌɑːrˈen/, /ˌɑːrˈen/]",[ C ] US,a \n \n registered nurse ; als...,注册护士（可用于姓名后）,"Alice Brody, RN",注册护士爱丽丝·布罗第
0,RN,noun,"[/ˌɑːrˈen/, /ˌɑːrˈen/]",UK,abbreviation for Royal Navy: used especially a...,英国皇家海军（尤用于海军军官姓名后，Royal Navy的缩写）,"Captain H. Doughty, RN",英国皇家海军上校 H.道蒂
1,R.N.,noun [ C ],"[/ˌɑːrˈen/, /ˌɑːrˈen/]",,abbreviation for\n \n registere...,注册护士（registered nurse的缩写）,,
2,riyal,noun [ C ],"[/riˈɑːl/, /riːˈjɑːl/]",,the standard unit of money used in Saudi Arabi...,里亚尔（沙特阿拉伯和卡塔尔的货币单位）,,


# 清洗（Cleaning）

In [124]:
import re

def clean(en_definition: str) -> str:
    # 去除多余的行
    # Remove unnecessary new lines
    en_definition = re.sub(r"\s*\n", " ", en_definition)
    
    # 去除没必要的空格
    # Remove unnecessary spaces
    result: str = ""
    str_list = re.split(r"\s+", en_definition)
    str_list_len = len(str_list)
    for i, sub_str in enumerate(str_list):
        if i == str_list_len - 1:
            result += str_list[-1]
        else:
            result += sub_str + " "
    return result

words_df["part_of_speech"] = words_df["part_of_speech"].apply(clean)
words_df["definition_en"] = words_df["definition_en"].apply(clean)
words_df["definition_zh"] = words_df["definition_zh"].apply(clean)
words_df["example_en"] = words_df["example_en"].apply(clean)
words_df["example_zh"] = words_df["example_zh"].apply(clean)
words_df["cefr"] = words_df["cefr"].apply(clean)
words_df.head(100)

,word,part_of_speech,phonetic_symbol,cefr,definition_en,definition_zh,example_en,example_zh
0,RN,noun,"[/ˌɑːrˈen/, /ˌɑːrˈen/]",[ C ] US,a registered nurse ; also used after the name ...,注册护士（可用于姓名后）,An RN makes about $30 an hour in this city.,在这座城市里一名注册护士的时薪约为30美元。
0,RN,noun,"[/ˌɑːrˈen/, /ˌɑːrˈen/]",[ C ] US,a registered nurse ; also used after the name ...,注册护士（可用于姓名后）,"Alice Brody, RN",注册护士爱丽丝·布罗第
0,RN,noun,"[/ˌɑːrˈen/, /ˌɑːrˈen/]",UK,abbreviation for Royal Navy: used especially a...,英国皇家海军（尤用于海军军官姓名后，Royal Navy的缩写）,"Captain H. Doughty, RN",英国皇家海军上校 H.道蒂
1,R.N.,noun [ C ],"[/ˌɑːrˈen/, /ˌɑːrˈen/]",,abbreviation for registered nurse,注册护士（registered nurse的缩写）,,
2,riyal,noun [ C ],"[/riˈɑːl/, /riːˈjɑːl/]",,the standard unit of money used in Saudi Arabi...,里亚尔（沙特阿拉伯和卡塔尔的货币单位）,,
...,...,...,...,...,...,...,...,...
45,riser,noun,"[/ˈraɪ.zər/, /ˈraɪ.zɚ/]",[ C ],something such as a share price or currency wh...,上涨之物（如股价、汇率等）,"Old Mutual , the financial services group, was...",金融服务公司英国耆卫保险，是富时100指数中股价上涨最多的。
46,rise to the occasion/challenge,idiom,"[, ]",,to show that you can deal with a difficult sit...,成功应对困难；临危不乱,In the exam she rose to the occasion and wrote...,在考试中，她临场发挥写出了一篇非常出色的文章。
47,rise to the bait,idiom,"[, ]",,to accept an offer or suggestion that seems go...,上钩，上当,"They offered a good salary, but I didn't rise ...",他们以优厚的工资条件作诱饵，不过我没有上当。
48,rise to fame,idiom,"[, ]",,to become famous,成名，出名,He rose to fame in the 90s as a TV presenter.,他在90年代作为电视节目主持人成名。
